# Generic processor integration procedure

This notebook is the main template for integration tests of DPR processors.   
Use it for the step 2 of the DPR integration procedure: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/1.+DPR+integration+PROCEDURE   
This step covers the basic tests of a processor, by running it outside of any workflow to make sure the release tested doesn't contain any blocking bug and is suitable for an usage in a more complex environment.

*Before running this notebook*, you need to perform the step 1 of the procedure, and in particular **retrieve the test payload** you are going to use (often available in the Git repository of the processor you are working with) and **identify the test data** you will use for this integration test. For this test data, you will have to **manually add it to the s3 bucket**, though an optional staging step or a more basic copy/paste, and **manually add the files location in the test payload**.   

Do not hesitate to copy/paste this notebook in a dedicated sprint demo folder if needed, and adapt it to the processor you are running for this demo.   

**IMPORTANT:** Please keep this notebook up-to-date as it will be used throughout the duration of the project.

## STEP 1 - Demo initialization and Dask cluster creation

This step initializes the demo and creates the Dask cluster for the chosen processor.   
You can change the "Optional change" parts as you need, but we do not recommend changing other parts of the code.   

Run all cells in this section.

### 1.1 - Setup variables

Set up the configuration needed to spawn the Dask cluster for a processor.   
Choose the processor you want to test.   
You can also use the experimental configuration provided for fine debugging on local mode, and use a specific tag if you are working on a development branch.   

**Warning:** If you are testing a new processor, don't forget to update *rs-dpr-service*, *rs-client* and *rs-demo* beforehand so the processor appears in the list

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.widget_utils import *

# Choose processor to create a Dask cluster for
dpr_proc_radio

In [ ]:
# Experimental DPR processor configuration, used only for testing.
# See: https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/utils/settings.py
# The local cluster config can be used to debug a processor using the rs-dpr-service image in debug mode,
# in local mode ONLY

# === OPTIONAL CHANGE ===
# Set the "enabled" field to True to use the localcluster config
# and tune the parameters as you want
experimental_config = {
    "local_cluster": {
        "enabled": False, # Use False to disable
        "n_workers": 4,
        "memory_limit": "58GiB",
    },
    "local_files": {
        "local_dir": None, #"/tmp/data", # Use None to disable
        "overwrite_input": False,
        "upload_output": True,
    },
}

In [ ]:
# "latest" tag is (by default) the latest image released by "rs-workflow-env" repository for this processor.
# If your image is on a development branch, use the associated tag for this image instead of "latest" in the line below.
# Example:
# processor_image_tag = "feat-rspy999-my-dev-branch"

# === OPTIONAL CHANGE ===
# Specify a tag for the image to use
processor_image_tag = "latest"

### 1.2 - Create Dask cluster

Create a new Dask cluster for the processor chosen in the previous step.   
The image used is adapted to your environment (local or cluster) and the tag specified in the previous step.   

**IMPORTANT:** For any **new** processor added, add the corresponding "case" in the cell below, following what is done for the existing ones.

In [ ]:
# Which mode for the processor image
if local_mode:
    target = "local"
else:
    target = "k8s"

# Init demo
init_demo()

# Init Dask cluster for chosen processor
match dpr_proc_radio.value:
    case "mockup":
        init_dask_cluster_mockup(scale=1)
    case DprProcessor.S1L0.value | DprProcessor.S3L0.value:
        init_dask_cluster_l0(
            image=f"ghcr.io/rs-python/dask/l0/{target}:{processor_image_tag}",
            scale=1,
            big_resources=True,  # provide more ram and cpu
            worker_cores=4,      # number of CPU per worker 
            worker_memory=58,    # memory per worker in GB
        )
    case DprProcessor.S1ARD.value:
        init_dask_cluster_s1ard(
            image=f"ghcr.io/rs-python/dask/s1ard/{target}:{processor_image_tag}",
            scale=1,
            big_resources=True,  # provide more ram and cpu
            worker_cores=4,      # number of CPU per worker 
            worker_memory=58,    # memory per worker in GB
        )
    case DprProcessor.S3OLCI.value:
        init_dask_cluster_s3olci(
            image=f"ghcr.io/rs-python/dask/s3olci/{target}:{processor_image_tag}",
            scale=1,
            big_resources=True,  # provide more ram and cpu
            worker_cores=4,      # number of CPU per worker 
            worker_memory=58,    # memory per worker in GB
        )
    # === OPTIONAL CHANGE ===
    # Add here any new processor to the list

In [ ]:
# Reload the global vars again
from resources.utils import *
from resources.dask_utils import *

# Other imports
import os.path as osp
from IPython.display import JSON
from resources.dpr_utils import DprDemo
from rs_client.ogcapi.dpr_client import DprProcessor

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)